# 2교시. OCR 기반 텍스트 추출 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/02_ocr_basic.ipynb)

**목표:** OCR 결과의 텍스트·위치·신뢰도를 읽고 원본과 비교합니다.

**결과물:** `ocr_result.json`

- 모든 필수 실습은 Google Colab에서 진행합니다.
- API 키나 결제가 필요 없습니다.
- 식별정보를 가린 교육용 샘플 한 장만 사용합니다.
- 개인·회사 문서를 외부 API에 보내거나 공개 Streamlit 주소를 만들지 않습니다.
- 실행이 3분을 넘으면 중지하고 준비 결과를 선택합니다.
- 필요한 셀을 위에서 아래로 다시 실행하고, 계속 실패하면 강사에게 알립니다.
- 실습이 끝나면 Colab 출력과 런타임 파일을 삭제하고 런타임을 종료합니다.


In [ ]:
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
import base64
import io
from PIL import Image, ImageDraw

SAMPLE_IMAGE_BASE64 = '...'
receipt_image = Image.open(
    io.BytesIO(base64.b64decode(SAMPLE_IMAGE_BASE64))
).convert("RGB")
MOCK_OCR_RESULT = [{'box': [[80, 70], [330, 70], [330, 125], [80, 125]], 'text': '샘플문구점', 'confidence': 0.98}, {'box': [[80, 160], [500, 160], [500, 205], [80, 205]], 'text': '거래일자: 2026-07-27', 'confidence': 0.97}, {'box': [[80, 270], [650, 270], [650, 315], [80, 315]], 'text': '연필 2개 × 1,000원 = 2,000원', 'confidence': 0.93}, {'box': [[80, 340], [650, 340], [650, 385], [80, 385]], 'text': '노트 1개 × 3,000원 = 3,000원', 'confidence': 0.95}, {'box': [[80, 465], [440, 465], [440, 520], [80, 520]], 'text': '합계: 5,000원', 'confidence': 0.99}]
OCR_RESULT = MOCK_OCR_RESULT
SOURCE_MODE = "prepared"
print("준비된 OCR 결과:", len(OCR_RESULT), "개 영역")


## 핵심 3개

1. OCR 결과는 텍스트·위치·신뢰도를 포함할 수 있습니다.
2. 흐림과 기울기는 오류 가능성을 높입니다.
3. 신뢰도는 정답이 아니라 검토 신호입니다.


In [ ]:
def draw_boxes(image, result):
    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)
    for item in result:
        xs = [point[0] for point in item["box"]]
        ys = [point[1] for point in item["box"]]
        draw.rectangle(
            (min(xs), min(ys), max(xs), max(ys)),
            outline="#167D7F",
            width=4,
        )
    return annotated

annotated = draw_boxes(receipt_image, OCR_RESULT)
annotated_path = OUTPUT_DIR / "ocr_boxes.png"
annotated.save(annotated_path)
print("바운딩 박스 이미지:", annotated_path)


## 기본 실습. PaddleOCR 3.7 + PP-OCRv5 Korean

한국어 인식은 `lang="korean"`과 `PP-OCRv5`를 사용합니다.
아래 값을 `True`로 바꾸어 실행합니다. 설치·다운로드·실행이
3분 안에 끝나지 않으면 중지하고 준비 결과로 같은 원본 대조 실습을 계속합니다.


In [ ]:
RUN_PADDLEOCR = False

if RUN_PADDLEOCR:
    import subprocess

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "paddlepaddle==3.2.1",
        ]
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "paddleocr==3.7.0"]
    )
    from paddleocr import PaddleOCR

    image_path = OUTPUT_DIR / "receipt_sample.png"
    receipt_image.save(image_path)
    pipeline = PaddleOCR(
        lang="korean",
        ocr_version="PP-OCRv5",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        device="cpu",
    )
    live_pages = list(pipeline.predict(str(image_path)))
    live_payload = live_pages[0].json
    if callable(live_payload):
        live_payload = live_payload()
    live_result = live_payload.get("res", live_payload)
    texts = live_result.get("rec_texts", [])
    boxes = live_result.get("rec_polys", [])
    scores = live_result.get("rec_scores", [])
    OCR_RESULT = [
        {
            "box": box,
            "text": text,
            "confidence": float(score),
        }
        for box, text, score in zip(boxes, texts, scores)
    ]
    SOURCE_MODE = "live_paddleocr"
    print("LIVE PaddleOCR 텍스트:", texts)
else:
    print("준비된 OCR 결과로 원본 대조 실습을 계속합니다.")


## 실습. 원본 대조 표시와 함께 저장


In [ ]:
import json

reviewed = [
    {
        **item,
        "matches_source": None,
        "review_note": "",
    }
    for item in OCR_RESULT
]
output = {
    "source_mode": SOURCE_MODE,
    "items": reviewed,
}
output_path = OUTPUT_DIR / "ocr_result.json"
output_path.write_text(
    json.dumps(output, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("저장 완료:", output_path)


## 확인

- `ocr_result.json`이 생성됐는가?
- 높은 신뢰도도 원문과 비교했는가?
- 실제 OCR이 3분 안에 실행되지 않을 때 준비 결과로 전환했는가?
